# 19. 임계값 튜닝 & 모델 평가 (README 7, 8)

순서: 1.앙상블 로드 > 2.임계값 튜닝 > 3.행 단위 평가 > 4.개체 단위 롤링 평가


In [ ]:
import sys, os, json, joblib
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath(".."))

from src.eval_core import ThresholdTuner, evaluate_row_level, EntityLevelEvaluator
import config.eval_config as cfg
import config.train_config as tcfg
print("환경 준비 완료")


## 1. 학습된 앙상블 로드


In [ ]:
from pathlib import Path
SAVE_DIR = Path(tcfg.MODEL_SAVE_DIR)

with open(SAVE_DIR / "feature_cols.json", encoding="utf-8") as f:
    FEATURE_COLS = json.load(f)

models = [joblib.load(p) for p in sorted(SAVE_DIR.glob("subset_*.pkl"))]
print(f"로드: {len(models)}개 서브셋 모델  피처: {len(FEATURE_COLS)}개")

class _EnsembleInfer:
    def __init__(self, models): self.models = models
    def predict_proba(self, df, feature_cols):
        return np.mean([m.predict_proba(df[feature_cols])[:,1] for m in self.models], axis=0)

ensemble = _EnsembleInfer(models)


## 2. 임계값 튜닝 (val_calib)


In [ ]:
df_calib = pd.read_parquet(cfg.VAL_CALIB_PATH)
y_calib_prob = ensemble.predict_proba(df_calib, FEATURE_COLS)

tuner = ThresholdTuner(max_fpr=cfg.MAX_FPR, n_grid=cfg.THRESHOLD_N_GRID)
tuner_result = tuner.fit(df_calib[cfg.TARGET_COL].values, y_calib_prob)
tuner.plot()

THRESHOLD = tuner.best_threshold
print(f"최적 임계값: {THRESHOLD:.4f}")


## 3. 행 단위 평가 (test)


In [ ]:
df_test = pd.read_parquet(cfg.TEST_PATH)
y_test_prob = ensemble.predict_proba(df_test, FEATURE_COLS)

row_result = evaluate_row_level(
    df_test[cfg.TARGET_COL].values,
    y_test_prob,
    THRESHOLD,
    title="Test Set",
)


## 4. 개체 단위 롤링 평가 (test)


In [ ]:
ev = EntityLevelEvaluator(
    serial_col=cfg.SERIAL_COL,
    date_col=cfg.DATE_COL,
    target_col=cfg.TARGET_COL,
)
entity_result = ev.evaluate(df_test, y_test_prob, THRESHOLD)
EntityLevelEvaluator.plot(entity_result)


## 5. 미탐 개체 상세 확인


In [ ]:
ed = entity_result["entity_df"]
miss_list = ed[(ed["is_failure"]==1) & (ed["has_alarm"]==0)]
print(f"미탐 고장 개체 수: {len(miss_list)}")
miss_list.head(20)
